In [ ]:
import os, sys
sys.path.append('../')
import MeshFEM, mesh, benchmark
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [ ]:
import plot_video_utils

In [ ]:
base_path = 'local_exps_sm_omp_cholmod_0121/'

In [ ]:
user_model_name = 'bear_cut'

In [ ]:
hessian_option_list = ['Adaptive', 'Always', 'xbasedAlways', 'TinyAD']
thread_num_list = [2, 4, 8, 20]

In [ ]:
directory = os.path.join(base_path, user_model_name)
# Build nested-dictonary
model_dict = {}
for hessian_option in hessian_option_list:
    hessian_dict = {}
    for thread_num in thread_num_list:
        thread_dir_name = 'thread' + '_' + str(thread_num)
        cur_dir = os.path.join(directory, hessian_option, thread_dir_name)
        fast_ind = plot_video_utils.getFastestRepeatIndex(cur_dir) # read file 'summary.txt'
        repeat_dir_name = 'repeat' + '_' + fast_ind
        data_dir = os.path.join(cur_dir, repeat_dir_name)
        obj_arr, time_arr, grad_norm_arr, benchmark_dict = plot_video_utils.read_benchmark_data(data_dir)
        # build dictionary different for TinyAD
        thread_dict = {}
        thread_dict['iter'] = obj_arr.shape[0]
        thread_dict['time'] = time_arr[-1]
        if hessian_option == 'TinyAD':
            thread_dict['linsolve'] = benchmark.totalTime('Linear Solve$', d=benchmark_dict)
            thread_dict['hessian_eval'] = benchmark.totalTime('Hessian Evaluation$', d=benchmark_dict)
            thread_dict['line_search'] = benchmark.totalTime('Line Search$', d=benchmark_dict)
        else:
            thread_dict['linsolve'] = benchmark.totalTime('CholeskyFactorizerBase.solve$', d=benchmark_dict)
            thread_dict['hessian_eval'] = benchmark.totalTime('NewtonMultiobjectiveProblem.hessian$', d=benchmark_dict)
            thread_dict['symbol'] = benchmark.totalTime('Catamari Symbolic Factorize$', d=benchmark_dict)
            thread_dict['numeric'] = benchmark.totalTime('Catamari Numeric Factorize$', d=benchmark_dict)
        hessian_dict[thread_num] = thread_dict
    model_dict[hessian_option] = hessian_dict

In [ ]:
model_dict

# draw Bar Plots comparing different thread's timings

In [ ]:
compare_key = 'linsolve'

In [ ]:
hessian_option_list = ['Adaptive', 'Always', 'xbasedAlways']
thread_num_list = [2, 4, 8, 20]

In [ ]:
mywidth = 0.2
a = np.arange(len(thread_num_list))
num_options = len(hessian_option_list)

fig, ax = plt.subplots(figsize=(15,8))
for hessian_ind, hessian_option in enumerate(hessian_option_list):
    metric_list = [model_dict[hessian_option][thread_num][compare_key] for thread_num in thread_num_list]
    position = a + (hessian_ind - num_options/2) * mywidth + mywidth / 2
    ax.bar(position, metric_list, width=mywidth, label=hessian_option)

ax.set_xticks(a)
ax.set_xticklabels(thread_num_list)

ax.set_xlabel('Number of Threads')
ax.set_ylabel(compare_key.capitalize())
ax.set_title('Comparison of Metrics')
ax.legend(loc='upper right')

plt.show()